# Part 3: GAPMAP Extensions (Notebook)

This notebook is the **Part 3 continuation of Part 2**.

- **Part 2 (baseline) = explicit gaps**: extract sentences where authors *explicitly state* a knowledge gap (e.g., “remains unclear…”) and evaluate against a manual gold standard with ROUGE-L.
- **Part 3 (extensions) = improve the Part 2 evaluation and add implicit gaps**:
  - **Explicit-focused extensions** (not “redoing Part 2,” but making evaluation fairer and analysis stronger):
    - `clean-gold`: remove citation-like junk from explicit gold sentences
    - `semantic-compare`: credit paraphrases (ROUGE can miss correct paraphrases)
    - `export-agreement`: find chunks where multiple models agree (high-confidence candidates)
  - **Implicit-gap capability (GAPMAP’s harder contribution)**: infer **implicit** knowledge gaps that are not stated directly.

## What you run in this notebook

- **Gold hygiene:** `clean-gold`
- **Semantic-augmented eval:** `semantic-compare`
- **Triple-model agreement:** `export-agreement`
- **Implicit-gap scaffolding (for human labeling if you want an implicit gold set):** `implicit-template`, `implicit-sample`
- **TABI batch (GAPMAP-style implicit gaps):** `tabi-extract` → structured **Claim / Grounds / Warrant / Bucket** per chunk
- **Entailment validation (quality screen, not a human gold standard):** `tabi-validate` → `mnli_entailment_prob` / `mnli_entailment_pass`
- **High-confidence export:** `tabi-high-confidence` → calibrated implicit gaps using MNLI threshold + bucket
- **View results:** Section **6b** loads `tabi_outputs*.csv` and shows tables in the notebook.

**Important:** Set `OPENAI_API_KEY` in the next cell *before* TABI or implicit-extract. Terminal runs do **not** see keys you only typed in another notebook unless you export them in that shell.

## 1. API keys and project root

Uncomment and paste your key, or rely on `OPENAI_API_KEY` already set in the kernel environment.

In [ ]:
import os
from pathlib import Path

# --- API key loading (do NOT paste your key into this notebook) ---
# Set OPENAI_API_KEY in your shell before launching Jupyter, e.g.
#   export OPENAI_API_KEY="sk-..."
# Or put it in a local .env file (which is .gitignored).
os.environ.setdefault("OPENAI_MODEL", "gpt-4o")

k = os.environ.get("OPENAI_API_KEY", "").strip()
print("OPENAI_API_KEY present:", bool(k and k.startswith("sk-")), "| len:", len(k))
print("OPENAI_MODEL:", os.environ.get("OPENAI_MODEL", "(default gpt-4o-mini in scripts)"))

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / "run_part3.py").exists(), "Start Jupyter from the project folder (contains run_part3.py)"
assert (PROJECT_ROOT / "Dataset").exists() or os.environ.get("GAPMAP_DATASET"), "Need Dataset/ or set GAPMAP_DATASET to your PDF folder"
print("PROJECT_ROOT:", PROJECT_ROOT)


## 2. Helper: run `run_part3.py` from the project root

Uses `.venv/bin/python` if present; otherwise `python` on your PATH.

In [2]:
import subprocess
import sys

VENV_PY = PROJECT_ROOT / ".venv" / "bin" / "python"
PYTHON = str(VENV_PY) if VENV_PY.exists() else sys.executable


def run_part3(*args: str) -> int:
    """Invoke run_part3.py with subcommand + flags. Inherits os.environ (API keys)."""
    cmd = [PYTHON, str(PROJECT_ROOT / "run_part3.py"), *args]
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=os.environ.copy())
    return p.returncode


print("Using interpreter:", PYTHON)

Using interpreter: /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python


## 3. Gold cleaning, semantic compare, agreement export

In [3]:
run_part3("clean-gold")

$ /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python /Users/momoba/Desktop/Advanced ML Project /run_part3.py clean-gold


Wrote /Users/momoba/Desktop/Advanced ML Project /Part3_Output/gold_standard_cleaned.csv
Chunks with ≥1 gap after clean: 93 (citation-like sentences dropped: 7)


0

In [4]:
run_part3("semantic-compare", "--threshold", "0.55")

$ /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python /Users/momoba/Desktop/Advanced ML Project /run_part3.py semantic-compare --threshold 0.55


=== Semantic-augmented match: max(ROUGE-L, bow-cosine) >= 0.55 ===
Model                      Precision     Recall         F1
---------------------------------------------------------
gpt4o_mini                    0.5870     0.9213     0.7171
gpt_4o                        0.8839     0.9167     0.9000
ollama                        0.5351     0.8939     0.6695

Saved to /Users/momoba/Desktop/Advanced ML Project /Part3_Output/semantic_compare.txt
Note: This credits some paraphrases ROUGE alone would miss; compare to `python run_part2.py --compare`.


0

In [5]:
run_part3("export-agreement")

$ /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python /Users/momoba/Desktop/Advanced ML Project /run_part3.py export-agreement


Chunks with gaps in all three models: 99
Wrote /Users/momoba/Desktop/Advanced ML Project /Part3_Output/high_confidence_chunks.csv


0

## 4. Implicit-gap scaffolding (templates + optional one-chunk demo)

In [6]:
run_part3("implicit-template", "--n", "50")

$ /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python /Users/momoba/Desktop/Advanced ML Project /run_part3.py implicit-template --n 50


Wrote /Users/momoba/Desktop/Advanced ML Project /Part3_Output/implicit_gold_template.csv (50 rows). Fill implicit_gap_sentence manually.


0

In [7]:
run_part3("implicit-sample", "--n", "50")

$ /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python /Users/momoba/Desktop/Advanced ML Project /run_part3.py implicit-sample --n 50


Wrote /Users/momoba/Desktop/Advanced ML Project /Part3_Output/implicit_annotation_sample.csv (50 rows).


0

In [8]:
# Optional: paid API — one chunk implicit gap (set chunk_id)
CHUNK_ID = 21
if os.environ.get("OPENAI_API_KEY", "").strip().startswith("sk-"):
    run_part3("implicit-extract", "--chunk-id", str(CHUNK_ID))
else:
    print("Skip: set OPENAI_API_KEY in cell 1")

$ /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python /Users/momoba/Desktop/Advanced ML Project /run_part3.py implicit-extract --chunk-id 21


chunk_id=21
The study does not address whether the observed post-merger performance recovery would persist or change with even longer post-merger periods beyond the twenty rounds tested.


## 5. TABI batch (Claim / Grounds / Warrant / Bucket)

1. **`--dry-run`** — no API; writes schema sample (fast path check).
2. **Small live test** — `--max-chunks 10` after a valid key.
3. **Full corpus** — remove `--max-chunks`; add `--resume` if restarting.

Pre-flight auth is built into `run_part3.py`; bad keys stop before the full loop.

In [9]:
# No API — verify paths and CSV columns (~1 min: loads all PDFs)
run_part3(
    "tabi-extract",
    "--dry-run",
    "--max-chunks",
    "5",
    "--out",
    "Part3_Output/tabi_outputs_notebook_dryrun.csv",
)

$ /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python /Users/momoba/Desktop/Advanced ML Project /run_part3.py tabi-extract --dry-run --max-chunks 5 --out Part3_Output/tabi_outputs_notebook_dryrun.csv


DRY-RUN mode: no API calls; writing placeholder TABI JSON per chunk.
Wrote 5 new inference rows → Part3_Output/tabi_outputs_notebook_dryrun.csv (file now has 5 rows, one per chunk_id).



TABI (openai/dry-run (dry-run)): 100%|██████████| 5/5 [00:00<00:00, 34044.68it/s]


0

In [10]:
# Live test (requires real OPENAI_API_KEY in this kernel)
if os.environ.get("OPENAI_API_KEY", "").strip().startswith("sk-"):
    os.environ.setdefault("OPENAI_MODEL", "gpt-4o")
    run_part3(
        "tabi-extract",
        "--backend",
        "openai",
        "--max-chunks",
        "10",
        "--out",
        "Part3_Output/tabi_outputs_notebook_test.csv",
        "--sleep",
        "0.25",
    )
else:
    print("Skip: set OPENAI_API_KEY in cell 1")

$ /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python /Users/momoba/Desktop/Advanced ML Project /run_part3.py tabi-extract --backend openai --max-chunks 10 --out Part3_Output/tabi_outputs_notebook_test.csv --sleep 0.25


OpenAI pre-flight OK (gpt-4o).



TABI (openai/gpt-4o):   0%|          | 0/10 [00:00<?, ?it/s]


TABI (openai/gpt-4o):  10%|█         | 1/10 [00:01<00:12,  1.36s/it]


TABI (openai/gpt-4o):  20%|██        | 2/10 [00:03<00:16,  2.07s/it]

...


TABI (openai/gpt-4o):  70%|███████   | 7/10 [00:13<00:06,  2.00s/it]


TABI (openai/gpt-4o):  80%|████████  | 8/10 [00:16<00:04,  2.15s/it]


TABI (openai/gpt-4o):  90%|█████████ | 9/10 [00:18<00:02,  2.05s/it]

Wrote 10 new inference rows → Part3_Output/tabi_outputs_notebook_test.csv (file now has 10 rows, one per chunk_id).



TABI (openai/gpt-4o): 100%|██████████| 10/10 [00:19<00:00,  1.95s/it]


In [11]:
import pandas as pd
from IPython.display import display

p = PROJECT_ROOT / "Part3_Output" / "tabi_outputs_notebook_test.csv"
if not p.exists():
    p = PROJECT_ROOT / "Part3_Output" / "tabi_outputs.csv"
if p.exists():
    d = pd.read_csv(p)
    c = d["claim"].fillna("").astype(str).str.strip()
    good = ((c != "") & (c.str.upper() != "NONE")).sum()
    err = d["parse_error"].fillna("").astype(str).str.len().gt(0).sum()
    print("File:", p.name)
    print("non-empty claims (not NONE):", int(good))
    print("rows with parse_error text:", int(err))
    display(d.head(3))
else:
    print("No TABI CSV found yet — run the cells above.")

File: tabi_outputs_notebook_test.csv
non-empty claims (not NONE): 10
rows with parse_error text: 0


,chunk_id,article_filename,word_count,backend,model,claim,grounds,warrant,bucket,parse_error,raw_response_excerpt
0,0,Does-college-education-make-women-less-likely-...,971,openai,gpt-4o,The causal mechanisms by which higher educatio...,the extent to which this reflects a causal eff...,The text acknowledges challenges in establishi...,more_probable,NaN,"{""claim"":""The causal mechanisms by which highe..."
1,1,Does-college-education-make-women-less-likely-...,976,openai,gpt-4o,The impact of higher education expansion on ma...,this finding might not generalise to rural ori...,The text suggests that the positive effects ob...,more_probable,NaN,"{""claim"":""The impact of higher education expan..."
2,2,Does-college-education-make-women-less-likely-...,989,openai,gpt-4o,The impact of the hukou system on educational ...,The hukou (household registration) system...is...,While the text discusses educational assortati...,more_probable,NaN,"{""claim"":""The impact of the hukou system on ed..."


## 6. Optional: RoBERTa-MNLI entailment (`tabi-validate`)

Requires: `pip install torch transformers` (see `requirements_part3_optional.txt`).

In [12]:
TAB_IN = PROJECT_ROOT / "Part3_Output" / "tabi_outputs.csv"
if TAB_IN.exists():
    try:
        import torch  # noqa: F401
        import transformers  # noqa: F401
    except ImportError:
        print("Install: pip install torch transformers")
    else:
        run_part3("tabi-validate", "--input", str(TAB_IN), "--threshold", "0.4")
else:
    print("Missing", TAB_IN, "— run full tabi-extract first.")

$ /Users/momoba/Desktop/Advanced ML Project /.venv/bin/python /Users/momoba/Desktop/Advanced ML Project /run_part3.py tabi-validate --input /Users/momoba/Desktop/Advanced ML Project /Part3_Output/tabi_outputs.csv --threshold 0.4



Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7765.29it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

MNLI entailment:   0%|          | 0/1367 [00:00<?, ?it/s]


MNLI entailment:   0%|          | 3/1367 [00:02<12:12,  1.86it/s]


MNLI entailment:   1%|          | 7/1367 [00:02<04:26,  5.10it/s]


MNLI entailment:   1%|          | 11/1367 [00:02<02:42,  8.35it/s]

...


MNLI entailment:  99%|█████████▉| 1354/1367 [01:23<00:00, 17.03it/s]


MNLI entailment:  99%|█████████▉| 1358/1367 [01:23<00:00, 16.06it/s]


MNLI entailment: 100%|█████████▉| 1362/1367 [01:23<00:00, 16.11it/s]


MNLI entailment: 100%|██████████| 1367/1367 [01:23<00:00, 16.28it/s]


Entailment threshold 0.4: pass 919/1367 rows with non-empty claims
Wrote /Users/momoba/Desktop/Advanced ML Project /Part3_Output/tabi_outputs_validated.csv


## 6b. View TABI outputs here (no API)

Loads **`Part3_Output/tabi_outputs.csv`** and, if present, **`tabi_outputs_validated.csv`**. Run Section 5 (TABI) and Section 6 (`tabi-validate`) first, or open the CSVs in Cursor/Excel.

In [13]:
import pandas as pd
from IPython.display import display

TAB = PROJECT_ROOT / "Part3_Output" / "tabi_outputs.csv"
VAL = PROJECT_ROOT / "Part3_Output" / "tabi_outputs_validated.csv"

if TAB.exists():
    d = pd.read_csv(TAB)
    c = d["claim"].fillna("").astype(str).str.strip()
    print("===", TAB.name, "===")
    print("rows:", len(d))
    print("non-NONE claims:", int(((c != "") & (c.str.upper() != "NONE")).sum()))
    if "parse_error" in d.columns:
        print("rows with parse_error:", int(d["parse_error"].fillna("").astype(str).str.len().gt(0).sum()))
    display(d[["chunk_id", "article_filename", "bucket", "claim", "grounds"]].head(8))
else:
    print("Missing", TAB)

if VAL.exists():
    v = pd.read_csv(VAL)
    print("\n===", VAL.name, "===")
    if "mnli_entailment_pass" in v.columns:
        passed = int(v["mnli_entailment_pass"].fillna(False).astype(bool).sum())
        print("entailment_pass=True rows:", passed, "/", len(v))
    show = v[v["mnli_entailment_pass"] == True] if "mnli_entailment_pass" in v.columns else v
    cols = [c for c in ["chunk_id", "bucket", "claim", "grounds", "mnli_entailment_prob", "mnli_entailment_pass"] if c in show.columns]
    display(show[cols].head(10))
else:
    print("\n(No validated file yet — run Section 6 `tabi-validate`.)")

=== tabi_outputs.csv ===
rows: 1367
non-NONE claims: 1361
rows with parse_error: 0


,chunk_id,article_filename,bucket,claim,grounds
0,0,Does-college-education-make-women-less-likely-...,more_probable,The impact of educational assortative mating o...,the strong negative relationship observed betw...
1,1,Does-college-education-make-women-less-likely-...,more_probable,The impact of higher education expansion on ma...,this finding might not generalise to rural ori...
2,2,Does-college-education-make-women-less-likely-...,more_probable,The impact of the hukou system on educational ...,The hukou (household registration) system...is...
3,3,Does-college-education-make-women-less-likely-...,more_probable,The causal mechanisms linking higher education...,we provide direct causal evidence of the effec...
4,4,Does-college-education-make-women-less-likely-...,more_probable,The long-term effects of the HE expansion on e...,the massive HE expansion started in 1999...the...
5,5,Does-college-education-make-women-less-likely-...,more_probable,The long-term effects of the 1999 HE expansion...,The HE expansion came as a totally unanticipat...
6,6,Does-college-education-make-women-less-likely-...,more_probable,The differential impact of HE expansion on rur...,there is only a visible jump for urban student...
7,7,Does-college-education-make-women-less-likely-...,more_probable,The specific mechanisms by which education aff...,The LPM marginal effects of women ’ s educatio...



=== tabi_outputs_validated.csv ===
entailment_pass=True rows: 919 / 1367


,chunk_id,bucket,claim,grounds,mnli_entailment_prob,mnli_entailment_pass
1,1,more_probable,The impact of higher education expansion on ma...,this finding might not generalise to rural ori...,0.864524,True
2,2,more_probable,The impact of the hukou system on educational ...,The hukou (household registration) system...is...,0.678260,True
4,4,more_probable,The long-term effects of the HE expansion on e...,the massive HE expansion started in 1999...the...,0.903421,True
6,6,more_probable,The differential impact of HE expansion on rur...,there is only a visible jump for urban student...,0.939888,True
7,7,more_probable,The specific mechanisms by which education aff...,The LPM marginal effects of women ’ s educatio...,0.697516,True
8,8,more_probable,The causal mechanisms by which education influ...,the heterogeneous effect of education on the p...,0.933343,True
9,9,more_probable,The impact of the one-child policy on educatio...,critical birth cohort cut-off for HE expansion...,0.843200,True
10,10,more_probable,The impact of heterogeneous exposure to higher...,the interaction effects of childhood hukou sta...,0.680694,True
11,11,more_probable,The impact of higher education expansion on ma...,our own heterogeneous effect analysis showing ...,0.685252,True
13,13,more_probable,The impact of educational attainment on marria...,Key marriage characteristics by wife ’ s birth...,0.980134,True


In [ ]:
import pandas as pd
from IPython.display import display

HC = PROJECT_ROOT / "Part3_Output" / "tabi_high_confidence_mnli06.csv"
if HC.exists():
    h = pd.read_csv(HC)
    print("===", HC.name, "===")
    print("rows:", len(h))
    cols = [c for c in ["chunk_id", "article_filename", "bucket", "mnli_entailment_prob", "claim", "grounds", "warrant"] if c in h.columns]
    display(h[cols].head(10))
else:
    print("Missing", HC, "— run: .venv/bin/python run_part3.py tabi-high-confidence --mnli-threshold 0.6 --out Part3_Output/tabi_high_confidence_mnli06.csv")

## 7. Expanded corpus (optional)

Put extra PDFs in a folder and set `GAPMAP_DATASET` **before** starting the kernel, or in cell 1:

```python
os.environ["GAPMAP_DATASET"] = "Dataset_expanded"
```

Then restart the kernel (so `run_part2` / `run_part3` pick it up) and re-run TABI cells.